In [6]:
# === 1. Standard Library ===
import warnings
from functools import reduce

# === 2. Core Data Science ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# === 3. Scikit-learn (Preprocessing & Pipelines) ===
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import (
    train_test_split, 
    RandomizedSearchCV, 
    StratifiedKFold
)

# === 4. Scikit-learn (Models) ===
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier, 
    StackingClassifier
)
from sklearn.naive_bayes import GaussianNB # This was in your week 9 code, adding it

# === 5. Scikit-learn (Metrics) ===
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    confusion_matrix,
    classification_report,
    precision_recall_curve
)

# === 6. Scikit-learn (Utilities) ===
from sklearn.exceptions import ConvergenceWarning

# === 7. Settings ===
print("Libraries imported.")
# Suppress convergence warnings for cleaner output during tuning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

Libraries imported.


In [ ]:
### Week 5 Code ###

# Helper for neat table printing
def format_table(df, title):
    print(f"\n=== {title} ===")
    print(df.to_string(index=False))
    print("=" * (len(title) + 6))

print("Libraries imported.")

# ==============================================
# 2. LOAD AND PREPROCESS DATA (ONCE)
# ==============================================
# Assume these datasets are in memory
try:
    person = dataset_89533079_person_df.copy()
    measures = dataset_89533079_measurement_df.copy()
    survey = dataset_89533079_survey_df.copy()
    condition = dataset_89533079_condition_df.copy()
    drug = dataset_89533079_drug_df.copy()
    procedure = dataset_89533079_procedure_df.copy()

    # Convert all date columns ONCE
    measures['measurement_datetime'] = pd.to_datetime(measures['measurement_datetime'], errors='coerce')
    survey['survey_datetime'] = pd.to_datetime(survey['survey_datetime'], errors='coerce')
    person['date_of_birth'] = pd.to_datetime(person['date_of_birth'], errors='coerce')
    condition['condition_start_datetime'] = pd.to_datetime(condition['condition_start_datetime'], errors='coerce')
    drug['drug_exposure_start_datetime'] = pd.to_datetime(drug['drug_exposure_start_datetime'], errors='coerce')
    procedure['procedure_datetime'] = pd.to_datetime(procedure['procedure_datetime'], errors='coerce')
    
    print("All datasets loaded and dates preprocessed.")

except NameError:
    print("ERROR: Source datasets (e.g., 'dataset_89533079_person_df') not found in memory.")
    # In a real environment, you would exit or load from CSV here.
    # For this optimization, we'll assume they exist.
    pass


# ==============================================
# 3. BASELINE COHORT DEFINITION
# ==============================================

print("Defining baseline cohort...")
# Filter for plausible BMI measurements (concept ID 3038553, value 10-80)
bmi_all = measures[
    (measures['measurement_concept_id'] == 3038553) &
    (measures['value_as_number'].notna()) &
    (measures['value_as_number'] >= 10) &
    (measures['value_as_number'] <= 80)
].copy()

# Determine the earliest BMI as the baseline
baseline = (
    bmi_all.sort_values(['person_id', 'measurement_datetime'])
    .drop_duplicates('person_id', keep='first')
    [['person_id', 'measurement_datetime', 'value_as_number']]
    .rename(columns={'measurement_datetime': 'baseline_date',
                     'value_as_number': 'baseline_BMI'})
)

# Apply the BMI < 30 restriction for the cohort definition
# Renaming to 'baseline_bmi' to match usage in your later scripts
baseline_bmi = baseline[baseline['baseline_BMI'] < 30].copy()

# Add age at baseline
baseline_bmi = baseline_bmi.merge(person[['person_id', 'date_of_birth']], on='person_id', how='left')

# Calculate age in years
baseline_bmi['age_at_baseline'] = (baseline_bmi['baseline_date'] - baseline_bmi['date_of_birth']).dt.days / 365.25

# --- Define final cohort IDs and count ---
baseline_ids = baseline_bmi['person_id'].unique()
total_baseline_count = len(baseline_ids)

print(f"\n Baseline cohort defined: {total_baseline_count:,} participants.")

# Baseline cohort-level stats (mean & SD)
mean_bmi = baseline_bmi['baseline_BMI'].mean()
sd_bmi = baseline_bmi['baseline_BMI'].std()
mean_age = baseline_bmi['age_at_baseline'].mean()
sd_age = baseline_bmi['age_at_baseline'].std()

print("\n=== Baseline Cohort Summary ===")
print(f"Mean baseline BMI : {mean_bmi:.2f} (SD {sd_bmi:.2f})")
print(f"Mean baseline age : {mean_age:.1f} (SD {sd_age:.1f})")
print("===================================")


# ==============================================
# 4. HELPER FUNCTIONS FOR ANALYSIS
# ==============================================

def calculate_prevalence(data_df, id_column, concept_id_column, concept_dict, baseline_ids, domain_name):
    """
    Calculates prevalence of items in a concept dict against the baseline cohort.
    """
    summary = []
    total_baseline = len(baseline_ids)
    
    # Ensure baseline_ids is a set for fast lookups
    baseline_id_set = set(baseline_ids)
    
    for label, concept_ids in concept_dict.items():
        # Filter data to relevant concepts
        temp_df = data_df[data_df[concept_id_column].isin(concept_ids)]
        
        # Find unique person IDs *in the baseline cohort* who have the condition
        person_ids_with_item = set(temp_df[id_column])
        count = len(baseline_id_set.intersection(person_ids_with_item))
        
        pct = (count / total_baseline) * 100
        summary.append({'Domain': domain_name, 'Item': label, '%': round(pct, 1)})
    
    return pd.DataFrame(summary)

def summarize_sdoh(survey_df, concept_id, category_map, category_order, baseline_ids, unknown_val='Unknown'):
    """
    Categorizes and summarizes a survey question for the baseline cohort.
    """
    total_baseline = len(baseline_ids)
    
    # Get relevant data, drop duplicates (one answer per person in cohort)
    df = survey_df[
        (survey_df['question_concept_id'] == concept_id) &
        (survey_df['person_id'].isin(baseline_ids))
    ].drop_duplicates('person_id', keep='first').copy()
    
    df['answer_lower'] = df['answer'].astype(str).str.strip().str.lower()
    df['Category'] = unknown_val # Default
    
    for category, keywords in category_map.items():
        # 'keywords' should be a pipe-separated string for str.contains
        df.loc[df['answer_lower'].str.contains(keywords, na=False), 'Category'] = category
    
    # Group by new category and count unique people
    summary = df.groupby('Category')['person_id'].nunique().reset_index(name='Count')
    
    # Ensure all categories are present, even with 0 count
    summary_cat = pd.DataFrame({'Category': category_order})
    summary = summary_cat.merge(summary, on='Category', how='left').fillna(0)
    
    # Calculate % of *total baseline cohort*
    summary['Percent'] = (summary['Count'] / total_baseline * 100).round(1)
    
    # Fix dtypes after merge
    summary['Count'] = summary['Count'].astype(int)
    summary['Percent'] = summary['Percent'].astype(float).round(1)
    
    return summary[['Category', 'Count', 'Percent']]

print("Helper functions defined.")

# ==============================================
# 5. RUN COHORT ANALYSES
# ==============================================

# --- 5a: Demographics ---
print("\nCalculating demographics...")
# Filter survey to baseline cohort, dropping duplicates (one answer per person)
survey_baseline = survey[survey['person_id'].isin(baseline_ids)]
sex_df = survey_baseline[survey_baseline['question_concept_id'] == 1585845].drop_duplicates('person_id')
race_df = survey_baseline[survey_baseline['question_concept_id'] == 1586140].drop_duplicates('person_id')

# Calculate % against TOTAL baseline cohort
female_pct = (sex_df['answer'].str.lower().str.contains('female', na=False)).sum() / total_baseline_count * 100

def race_percent(keyword):
    return (race_df['answer'].str.lower().str.contains(keyword, na=False)).sum() / total_baseline_count * 100

summary_table = pd.DataFrame({
    "Variable": [
        "Sex at birth – Female",
        "Race/Ethnicity – White",
        "Race/Ethnicity – Black",
        "Race/Ethnicity – Hispanic",
        "Race/Ethnicity – Asian"
    ],
    "Type": [
        "Categorical (Binary)",
        "Categorical",
        "Categorical",
        "Categorical",
        "Categorical"
    ],
    "Summary Statistic": [
        f"{female_pct:.1f}%",
        f"{race_percent('white'):.1f}%",
        f"{race_percent('black'):.1f}%",
        f"{race_percent('hispanic'):.1f}%",
        f"{race_percent('asian'):.1f}%"
    ]
})
format_table(summary_table, "Demographics Summary")


# --- 5b: Anthropometrics ---
# The original code block below is commented out as 'anthro_df' is not defined.
# If 'anthro_df' was meant to be 'baseline_bmi', you could uncomment and adapt it.
"""
print("\n=== Cleaned Anthropometric Summary ===")
summary_data = {
    "Metric": ["BMI", "Weight (kg)", "Height (cm)"],
    "Mean": [
        round(anthro_df["BMI"].mean(), 2),
        round(anthro_df["Weight_kg"].mean(), 1),
        round(anthro_df["Height_cm"].mean(), 1)
    ],
    "SD": [ ... ],
    "Min": [ ... ],
    "Max": [ ... ],
    "Count": [ ... ]
}
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print(f"\nTotal participants: {len(anthro_df):,}")
print("=======================================")
"""

# --- 5c: Comorbidities ---
print("\nCalculating comorbidities...")
comorbidity_concepts = {
    'Hypertension': [316866],
    'Type 2 Diabetes Mellitus': [201826],
    'Hyperlipidemia': [432867],
    'Depression': [440383],
    'Sleep Apnea': [313459],
    'Hypothyroidism': [140673]
}
comorbidity_df = calculate_prevalence(
    data_df=condition,
    id_column='person_id',
    concept_id_column='condition_concept_id',
    concept_dict=comorbidity_concepts,
    baseline_ids=baseline_ids,
    domain_name='Comorbidity'
)
format_table(comorbidity_df, "Comorbidities Summary")


# --- 5d: Drug Exposures ---
print("\nCalculating drug exposures...")
drug_concepts = {
    'Systemic corticosteroids': [21602722],
    'Insulins and analogues': [21600713],
    'Antidepressants (Citalopram)': [797617],
    'Antipsychotics (Olanzapine)': [785788],
    'Anti-obesity drugs (Orlistat)': [741530]
}
drug_df = calculate_prevalence(
    data_df=drug,
    id_column='person_id',
    concept_id_column='drug_concept_id',
    concept_dict=drug_concepts,
    baseline_ids=baseline_ids,
    domain_name='Drug Exposure'
)
format_table(drug_df, "Drug Exposures Summary")


# --- 5e: Procedures ---
print("\nCalculating procedures...")
procedure_concepts = {
    'Bariatric surgery (any type)': [4326683]
}
procedure_df = calculate_prevalence(
    data_df=procedure,
    id_column='person_id',
    concept_id_column='procedure_concept_id',
    concept_dict=procedure_concepts,
    baseline_ids=baseline_ids,
    domain_name='Procedure'
)
format_table(procedure_df, "Procedures Summary")


# --- 5f: Social Determinants of Health (SDoH) ---
print("\nCalculating SDoH variables...")

# 1. Education
edu_map = {
    '≤ High school': 'twelve|ged|less than a high school',
    'Some college': 'college one to three',
    '≥ College': 'college graduate|advanced degree'
}
edu_order = ['≤ High school', 'Some college', '≥ College', 'Unknown']
edu_summary = summarize_sdoh(survey, 1585940, edu_map, edu_order, baseline_ids)
format_table(edu_summary, "Education Level Summary (Low → High)")

# 2. Income
income_map = {
    '<$25K': 'less 10k|10k 25k',
    '$25–50K': '25k 35k|35k 50k',
    '>$50K': '50k 75k|75k 100k|100k 150k|150k 200k|more 200k'
}
income_order = ['<$25K', '$25–50K', '>$50K', 'Unknown']
income_summary = summarize_sdoh(survey, 1585375, income_map, income_order, baseline_ids)
format_table(income_summary, "Household Income Summary (Low → High)")

# 3. Food Insecurity
food_map = {
    'Food Insecure': 'yes|often true|sometimes true',
    'Food Secure': 'no|never true'
}
food_order = ['Food Secure', 'Food Insecure', 'Unknown']
food_summary = summarize_sdoh(survey, 40192426, food_map, food_order, baseline_ids)
format_table(food_summary, "Food Insecurity Summary (Secure → Insecure)")

# 4. Physical Activity
activity_map = {
    'Low': 'disagree|strongly disagree|no',
    'Moderate': 'neutral|somewhat agree|sometimes',
    'High': 'agree|strongly agree|yes'
}
activity_order = ['Low', 'Moderate', 'High', 'Unknown']
activity_summary = summarize_sdoh(survey, 40192410, activity_map, activity_order, baseline_ids)
format_table(activity_summary, "Physical Activity Summary (Low → High)")

# 5. Housing (Represented as 0 per original script)
housing_summary = pd.DataFrame({
    'Category': ['Stable Housing', 'Unstable Housing', 'Unknown'],
    'Count': [0, 0, 0],
    'Percent': [0.0, 0.0, 0.0]
})
format_table(housing_summary, "Housing Instability Summary (No Usable Data)")

print("\n\n All analyses complete.")

In [ ]:
### Week 6 Code ###

# ==============================================
# 2. BUILD CONDITION FLAGS (Predictor)
# ==============================================
print("Building condition flags...")
# (Assuming 'condition' and 'baseline_bmi' are in memory)
# Make a working copy
cond = condition.copy()

# Attach baseline date and restrict to: [baseline - 6 months, baseline]
cond_base = cond.merge(baseline_bmi[['person_id', 'baseline_date']], on='person_id', how='inner')
cond_base['window_start'] = cond_base['baseline_date'] - pd.DateOffset(months=6)

in_window = (
    (cond_base['condition_start_datetime'] >= cond_base['window_start']) &
    (cond_base['condition_start_datetime'] <= cond_base['baseline_date'])
)
cond_base = cond_base.loc[in_window].copy()
cond_base['standard_concept_name'] = cond_base.get('standard_concept_name', "").fillna("")

# Define condition groups
conditions_of_interest = {
    "cond_hypertension": ["Hypertensive disorder", "Essential hypertension"],
    "cond_type_2_diabetes_mellitus": ["Type 2 diabetes mellitus"],
    "cond_hyperlipidemia": ["Hyperlipidemia"],
    "cond_depression": ["Depressive disorder", "Major depressive disorder"],
    "cond_sleep_apnea": ["Sleep apnea"],
    "cond_hypothyroidism": ["Hypothyroidism"],
}

# --- OPTIMIZED PIVOT METHOD (replaces loop) ---
flag_list = []
for label, keywords in conditions_of_interest.items():
    pattern = "|".join(keywords)
    mask = cond_base['standard_concept_name'].str.contains(pattern, case=False, na=False)
    
    # Get all person_ids who match, drop duplicates, and assign the label
    matched_ids = cond_base.loc[mask, 'person_id'].drop_duplicates()
    flag_df = pd.DataFrame({
        'person_id': matched_ids,
        'condition_label': label,
        'flag': 1
    })
    flag_list.append(flag_df)

if flag_list:
    # Combine all matched flags
    all_flags = pd.concat(flag_list, ignore_index=True)
    
    # Pivot to create the wide dataframe
    cond_flags = all_flags.pivot(
        index='person_id',
        columns='condition_label',
        values='flag'
    ).fillna(0).astype(int).reset_index()
else:
    # Create empty frame if no matches at all
    cond_flags = pd.DataFrame({'person_id': pd.Series(dtype='int64')})

print(" Condition flags created.")

# ==============================================
# 3. BUILD SDOH & DEMOGRAPHIC FLAGS (Predictor)
# ==============================================
print("Building SDoH and demographic predictors...")
# (Assuming 'survey' and 'person' are in memory)

# --- A. Education (concept_id = 1585940) ---
edu_df = survey.loc[
    survey["question_concept_id"].isin([1585940]), 
    ["person_id", "survey_datetime", "answer"]
].copy()
edu_df = edu_df.sort_values(["person_id", "survey_datetime"]).drop_duplicates("person_id", keep="last")
ans = edu_df["answer"].astype(str).str.strip().str.lower()
edu_flag = pd.Series(pd.NA, index=edu_df.index, dtype="Float64")
edu_flag = edu_flag.mask(ans.str.contains(r"college graduate or advanced degree", na=False), 1)
edu_flag = edu_flag.mask(ans.str.contains(r"twelve or ged|less than a high school|college one to three", na=False), 0)
edu_df["education_college_or_higher"] = edu_flag
edu_df = edu_df[["person_id", "education_college_or_higher"]]

# --- B. Food insecurity (40192517, 40192426) ---
food_ids = [40192517, 40192426]
food_df = survey.loc[
    survey["question_concept_id"].isin(food_ids),
    ["person_id", "answer"]
].copy()
food_df = (food_df.dropna(subset=["answer"])
           .drop_duplicates("person_id")
           .rename(columns={"answer": "food_insecurity"}))

# --- C. Housing instability (40192441; cutoff >= 2 moves) ---
housing_df = survey.loc[
    survey["question_concept_id"].isin([40192441]), 
    ["person_id", "answer"]
].copy()
housing_df["moves"] = pd.to_numeric(housing_df["answer"], errors="coerce")
housing_df["housing_instability"] = np.where(housing_df["moves"] >= 2, "Housing instability", "No housing instability")
housing_df = housing_df.drop_duplicates("person_id")[["person_id", "housing_instability"]]

# --- D. Demographics ---
demo_df = person.loc[:, ["person_id", "gender", "race", "ethnicity"]].drop_duplicates("person_id")

# --- E. Combine all predictors ---
# Start with the baseline cohort
predictor_dfs = [
    baseline_bmi,
    edu_df,
    food_df,
    housing_df,
    demo_df,
    cond_flags
]

# Use functools.reduce to cleanly merge all dataframes
predictor_df = reduce(lambda left, right: pd.merge(left, right, on='person_id', how='left'), predictor_dfs)

print(" Predictor table created.")

# ==============================================
# 4. BUILD OUTCOME VARIABLE (Dependent)
# ==============================================
print("Building outcome variable (incident obesity)...")
# (Assuming 'bmi_all' is in memory)

# 1) Attach baseline date
bmi_followup = bmi_all.merge(baseline_bmi[['person_id', 'baseline_date']], on='person_id', how='inner')

# 2) Keep only measurements > baseline and <= 1 year after
within_one_year = (
    (bmi_followup['measurement_datetime'] > bmi_followup['baseline_date']) &
    (bmi_followup['measurement_datetime'] <= bmi_followup['baseline_date'] + pd.DateOffset(days=365))
)
bmi_followup = bmi_followup.loc[within_one_year].copy()

# 3) Earliest post-baseline BMI per person
bmi_followup = (
    bmi_followup.sort_values(['person_id', 'measurement_datetime'])
    .drop_duplicates('person_id', keep='first')
    [['person_id', 'measurement_datetime', 'value_as_number']]
    .rename(columns={'measurement_datetime': 'followup_date', 'value_as_number': 'followup_BMI'})
)

# 4) Incident obesity flag
bmi_followup['incident_obesity'] = (bmi_followup['followup_BMI'] >= 30).astype(int)

print("✅ Outcome table created.")

# ==============================================
# 5. CREATE FINAL ANALYSIS DATASET
# ==============================================
print("Creating final analysis dataset...")

analysis_df = predictor_df.merge(
    bmi_followup[['person_id', 'followup_BMI', 'incident_obesity']],
    on='person_id',
    how='left'
)

# Filter to only people with a known outcome
analysis_df = analysis_df[analysis_df["incident_obesity"].notna()].copy()
analysis_df['incident_obesity'] = analysis_df['incident_obesity'].astype(int)


# ==============================================
# 6. OPTIMIZED ML PIPELINE
# ==============================================
print("Setting up and running ML Pipeline...")

# 1) Define Features (X) and Target (y)
target = 'incident_obesity'
drop_cols = [
    'person_id', 'baseline_date', 'date_of_birth', 
    'followup_BMI', 'incident_obesity', 'followup_date'
]
X = analysis_df.drop(columns=drop_cols, errors='ignore')
y = analysis_df[target]

# 2) Split data (stratify on target for imbalanced classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=45
)

# 3) Identify column types for preprocessing
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

# 4) Define Preprocessing Pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 5) Combine transformers into a single preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

# 6) Create the full model pipeline
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, solver='liblinear', random_state=45))
])

# 7) Fit the pipeline
print("Fitting model...")
clf.fit(X_train, y_train)
print("✅ Model fitted.")

# 8) Evaluate the model
y_proba = clf.predict_proba(X_test)[:, 1]
y_pred = clf.predict(X_test)

# === START: UPDATED METRICS ===

# Calculate metrics
auc = roc_auc_score(y_test, y_proba)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)


print("\n=== Model Performance (Optimized Pipeline) ===")
print(f"Test AUC:       {auc:.3f}")
print(f"Test Accuracy:  {acc:.3f}")
print(f"Test F1 Score:  {f1:.3f}")
print(f"Test Precision: {prec:.3f}")
print(f"Test Recall:    {rec:.3f}")
print("\nConfusion Matrix:")
print(cm)

In [ ]:
### Week 7 Code ###

# ==========================================================
# 2. CUSTOM TRANSFORMER FOR OUTLIER CLIPPING
# ==========================================================
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Clips outliers based on quantiles learned from the training data.
    """
    def __init__(self, quantiles=(0.01, 0.99)):
        self.quantiles = quantiles
        self.limits_ = {}

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        for col in X_df.columns:
            p1, p99 = X_df[col].quantile(self.quantiles)
            self.limits_[col] = (p1, p99)
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        for col in X_df.columns:
            p1, p99 = self.limits_[col]
            X_df[col] = X_df[col].clip(lower=p1, upper=p99)
        return X_df.values

# ==========================================================
# 3. LOAD AND CLEAN DATA
# ==========================================================
# ----- 0) Start from your assembled dataset -----
# Assuming 'analysis_df' is in memory
df = analysis_df.copy()
df = df[df["incident_obesity"].notna()].copy()

# Treat common invalid categorical tokens as missing
INVALID_TOKENS = {"", " ", "NA", "N/A", "NaN", "None", "null", "Missing", "Prefer not to answer"}
obj_cols = df.select_dtypes(include="object").columns
for c in obj_cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].mask(df[c].isin(INVALID_TOKENS), np.nan) # Replace with NaN

# ==========================================================
# 4. EDA (NON-LEAKING)
# ==========================================================
# This section is for *review only* and does not transform data, so it's safe.
print("=== Missing values per column (top 25) ===")
print(df.isnull().sum().sort_values(ascending=False).head(25))

# Quick scan for potential out-of-range values in numeric columns
num_cols_all = df.select_dtypes(include=np.number).columns.tolist()
print("\n=== Numeric columns scanned for potential outliers (IQR rule preview) ===")
for c in num_cols_all:
    s = df[c].dropna()
    if s.empty:
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    out_cnt = ((s < lo) | (s > hi)).sum()
    if out_cnt > 0:
        print(f"  - {c}: {out_cnt} potential outliers (IQR bounds ~ [{lo:.3g}, {hi:.3g}])")

# ==========================================================
# 5. DEFINE FEATURES, TARGET, AND SPLIT
# ==========================================================
# Drop non-predictive columns
X = df.drop(columns=["person_id", "followup_BMI", "incident_obesity"], errors="ignore")
y = df["incident_obesity"].astype(int)

# --- Handle datetime columns before splitting ---
datetime_cols = X.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns
if len(datetime_cols) > 0:
    print("\nConverting datetime columns to ordinal:", list(datetime_cols))
    for col in datetime_cols:
        X[col] = X[col].map(pd.Timestamp.toordinal)

# --- SPLIT DATA FIRST TO PREVENT LEAKAGE ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.20, random_state=45
)

# ==========================================================
# 6. BUILD PREPROCESSING PIPELINE
# ==========================================================
# Identify column types from the training data
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

# 

# --- Numeric Pipeline ---
# (Impute -> Clip Outliers -> Scale)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('clipper', Winsorizer(quantiles=(0.01, 0.99))),
    ('scaler', StandardScaler())
])

# --- Categorical Pipeline ---
# (Impute -> One-Hot Encode)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Fills NaN
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first')) # drop='first' matches original
])

# --- Combine pipelines ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

# ==========================================================
# 7) MODEL (Logistic Regression with CLASS WEIGHTS)
# ==========================================================

# Create the full pipeline: Preprocessing + Model
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        random_state=45,
        class_weight="balanced"  # <-- balances classes
    ))
])

# Train the entire pipeline
print("\nTraining model pipeline...")
clf.fit(X_train, y_train)
print("✅ Model training complete.")

# ==========================================================
# 8) EVALUATION with Threshold Optimization
# ==========================================================

# --- Predictions and probabilities (using the pipeline) ---
y_proba = clf.predict_proba(X_test)[:, 1]

# --- Find optimal threshold by maximizing F1 from precision–recall curve ---
prec, rec, thr = precision_recall_curve(y_test, y_proba)
# Add a small epsilon to avoid division by zero
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-9) 

# Handle boundary case where thr array is shorter
valid_f1_scores = f1_scores[:-1] if len(f1_scores) > len(thr) else f1_scores
best_idx = np.argmax(valid_f1_scores)
best_thresh = thr[best_idx]

print(f"\nOptimal Threshold (based on F1): {best_thresh:.4f}")

# --- Apply optimal threshold ---
y_pred = (y_proba >= best_thresh).astype(int)

# --- Compute metrics ---
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)
prec_score = precision_score(y_test, y_pred)
rec_score = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\n=== Model Performance (Pipeline + Optimized Threshold) ===")
print(f"Accuracy : {acc:.3f}")
print(f"AUC      : {auc:.3f}")
print(f"F1 Score : {f1:.3f}")
print(f"Precision: {prec_score:.3f}")
print(f"Recall   : {rec_score:.3f}")

print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
### Week 8 Code ###

# ==============================================
# Step 2: Load and Initial Cleanup
# ==============================================

# Drop ID/date columns that can cause data leakage
df = df.drop(columns=['person_id', 'baseline_date'], errors='ignore')

# Drop target leakage column followup_BMI (not to be used as predictor)
df = df.drop(columns=['followup_BMI'], errors='ignore')

# Identify and drop columns with 100% missing values
missing_all = df.columns[df.isnull().mean() == 1.0]
print("Columns with 100% missing values:", missing_all.tolist())

df = df.drop(columns=missing_all)
print(f"\n✅ Dropped {len(missing_all)} columns with 100% missing data.")

# ==============================================
# Step 3: Missingness EDA (Safe to do before split)
# ==============================================
print("\nRunning Missingness EDA...")
# 1) Summary tables
missing_counts = df.isna().sum().to_frame("missing_count")
missing_perc = (df.isna().mean()*100).round(2).to_frame("missing_percent")
summary = missing_counts.join(missing_perc).sort_values(by="missing_percent", ascending=False)
print(summary[summary['missing_percent'] > 0])

# 2) Missingness pattern
plt.figure(figsize=(8, 4))
plt.imshow(df.isna(), aspect='auto', cmap='gray_r', interpolation='nearest')
plt.xlabel("Variables")
plt.ylabel("Records")
plt.title("Missingness Matrix (white = missing)")
plt.tight_layout()
plt.show()

# 3) Percent missing by column
plt.figure(figsize=(10, 5))
summary["missing_percent"][summary["missing_percent"] > 0].sort_values(ascending=True).plot(kind="barh")
plt.xlabel("Percent Missing")
plt.title("Percent Missing by Column (where missing > 0%)")
plt.tight_layout()
plt.show()

# ==============================================
# Step 4: Define Target and Split Data (THE KEY!)
# ==============================================
# We split FIRST to prevent any data leakage

target = 'incident_obesity'

# Drop rows where the target is missing (if any)
df = df.dropna(subset=[target])
df[target] = df[target].astype(int)

print("\nClass distribution for target:")
print(df[target].value_counts(normalize=True).round(3))

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✅ Data split: {len(X_train)} train, {len(X_test)} test samples.")

# ==============================================
# Step 5: Build Imputation & Model Pipelines
# ==============================================
# Identify column types from the training set
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

# --- Pipeline 1: Simple Imputation ---
# Numeric: Impute with mean, then scale
num_transformer_simple = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
# Categorical: Impute with most frequent, then OHE
cat_transformer_simple = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
# Combine preprocessors
preprocessor_simple = ColumnTransformer(transformers=[
    ('num', num_transformer_simple, numeric_features),
    ('cat', cat_transformer_simple, categorical_features)
])
# Create final model pipeline
model_simple = Pipeline(steps=[
    ('preprocessor', preprocessor_simple),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# --- Pipeline 2: KNN Imputation ---
# This is a more advanced pipeline:
# 1. Scale numerics, OHE categoricals (to create a full numeric matrix)
# 2. THEN feed that matrix into KNNImputer
# 3. THEN feed the imputed matrix into the model

# Numeric: Just scale (imputation happens *after*)
num_transformer_knn = Pipeline(steps=[
    ('scaler', StandardScaler())
])
# Categorical: Just OHE (imputation happens *after*)
cat_transformer_knn = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
# Combine preprocessors
preprocessor_knn_base = ColumnTransformer(transformers=[
    ('num', num_transformer_knn, numeric_features),
    ('cat', cat_transformer_knn, categorical_features)
])
# Create final model pipeline
model_knn = Pipeline(steps=[
    ('preprocessor', preprocessor_knn_base),
    ('imputer', KNNImputer(n_neighbors=5)), # KNN runs *after* scaling/OHE
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

print("\n✅ Pipelines defined for Simple and KNN imputation.")

# ==============================================
# Step 6: Evaluate and Compare Models
# ==============================================
models_to_compare = {
    "Simple Imputation": model_simple,
    "KNN Imputation": model_knn
}

results = {}

for name, model in models_to_compare.items():
    print(f"\nTraining {name} model...")
    # Fit the entire pipeline on training data
    model.fit(X_train, y_train)
    
    # Predict on test data
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    # Store metrics
    results[name] = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs)
    }

results_df = pd.DataFrame(results).T

print("\n=== Model Performance Comparison (No Data Leakage) ===")
print(results_df.round(3))

# ==============================================
# Step 7: Visualization
# ==============================================
results_df.plot(kind='bar', figsize=(10, 6))
plt.title("Model Performance: Simple vs KNN Imputation (Leakage-Free)")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(title="Metrics", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
### Week 9 Code ###

# ==============================================
# Step 2: Load and Initial Cleanup
# ==============================================

# Drop known leakage or useless columns
df = df.drop(columns=['person_id', 'baseline_date', 'followup_BMI'], errors='ignore')

# Drop columns with 100% missing values
df = df.dropna(axis=1, how='all')

target = 'incident_obesity'

print("Remaining columns:", df.columns.tolist())
print("\nClass distribution:\n", df[target].value_counts(normalize=True))

# ==============================================
# Step 3: TRAIN/TEST SPLIT (Prevent Leakage)
# ==============================================
X = df.drop(columns=[target])
y = df[target]

# Identify numeric and categorical columns once based on the raw X
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✅ Data split successfully.")
print(f"Train shape: {X_train_raw.shape}, Test shape: {X_test_raw.shape}")

# ==============================================
# Step 4: Define Preprocessing Pipelines
# ==============================================
# This is the main optimization. We define the preprocessor objects
# but we don't apply them yet.

# --- Preprocessor 1: Simple Imputation ---
num_transformer_simple = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
cat_transformer_simple = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor_simple = ColumnTransformer(transformers=[
    ('num', num_transformer_simple, num_cols),
    ('cat', cat_transformer_simple, cat_cols)
])

# --- Preprocessor 2: KNN Imputation (for numerics) ---
# This matches your original logic: KNN for numeric, Simple for categorical
num_transformer_knn = Pipeline(steps=[
    ('scaler', StandardScaler()),  # Scale *before* KNN for distance metrics
    ('knn_imputer', KNNImputer(n_neighbors=5))
])
cat_transformer_knn = Pipeline(steps=[ # Categorical pipeline is same as simple
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor_knn = ColumnTransformer(transformers=[
    ('num', num_transformer_knn, num_cols),
    ('cat', cat_transformer_knn, cat_cols)
])

print("\n✅ Preprocessing pipelines defined.")

# ==============================================
# Step 5: Quick Comparison (Simple vs KNN)
# ==============================================
# We now compare the *full pipelines* on the raw data, preventing any leakage.

base_lr = LogisticRegression(max_iter=1000, class_weight='balanced')

# Build two complete pipelines to compare
pipe_simple = Pipeline([('preprocessor', preprocessor_simple), ('model', base_lr)])
pipe_knn = Pipeline([('preprocessor', preprocessor_knn), ('model', base_lr)])

print("\nComparing imputation pipelines...")
# Fit and score Simple pipeline
pipe_simple.fit(X_train_raw, y_train)
simple_auc = roc_auc_score(y_test, pipe_simple.predict_proba(X_test_raw)[:,1])

# Fit and score KNN pipeline
pipe_knn.fit(X_train_raw, y_train)
knn_auc = roc_auc_score(y_test, pipe_knn.predict_proba(X_test_raw)[:,1])

print(f"\nInitial Check - Simple Imputation AUC: {simple_auc:.4f}")
print(f"Initial Check - KNN Imputation AUC:    {knn_auc:.4f}")

# Select the BEST PREPROCESSOR *OBJECT*
if knn_auc > simple_auc:
    print("👉 Proceeding with KNN Imputation pipeline for tuning.")
    best_preprocessor = preprocessor_knn
else:
    print("👉 Proceeding with Simple Imputation pipeline for tuning.")
    best_preprocessor = preprocessor_simple

# ==============================================
# Step 6: Hyperparameter Tuning
# ==============================================
# We pass the *full pipeline* (preprocessor + model) into RandomizedSearchCV
# We fit on the *raw data* (X_train_raw, y_train)

print("\nStarting Hyperparameter Tuning... (this takes time!)")

# 1. Define Parameter Grids (note the 'model__' prefix)
param_grid_lr = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear'] # needed for l1
}

param_grid_rf = {
    'model__n_estimators': [100, 300, 500],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__class_weight': ['balanced', 'balanced_subsample', None]
}

param_grid_gb = {
    'model__n_estimators': [100, 300],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__max_depth': [3, 5, 7]
}

# 2. Define base pipelines for tuning
pipe_lr = Pipeline([
    ('preprocessor', best_preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

pipe_rf = Pipeline([
    ('preprocessor', best_preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

pipe_gb = Pipeline([
    ('preprocessor', best_preprocessor),
    ('model', GradientBoostingClassifier(random_state=42))
])

# 3. Set up Randomized Search objects
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

tune_lr = RandomizedSearchCV(pipe_lr, param_grid_lr, n_iter=5, cv=cv, scoring='roc_auc', n_jobs=-1, random_state=42)
tune_rf = RandomizedSearchCV(pipe_rf, param_grid_rf, n_iter=5, cv=cv, scoring='roc_auc', n_jobs=-1, random_state=42)
tune_gb = RandomizedSearchCV(pipe_gb, param_grid_gb, n_iter=5, cv=cv, scoring='roc_auc', n_jobs=-1, random_state=42)

# 4. Execute Tuning (on RAW data)
tune_lr.fit(X_train_raw, y_train)
print("✅ Logistic Regression Tuned")
tune_rf.fit(X_train_raw, y_train)
print("✅ Random Forest Tuned")
tune_gb.fit(X_train_raw, y_train)
print("✅ Gradient Boosting Tuned")

# ==============================================
# Step 7: Evaluate Tuned Models & Stacking
# ==============================================

# Get the best *pipeline* from each search
best_lr_pipe = tune_lr.best_estimator_
best_rf_pipe = tune_rf.best_estimator_
best_gb_pipe = tune_gb.best_estimator_

print("\n=== Best Parameters Found ===")
# best_params_ includes the 'model__' prefix
print("LR:", {k.split('__')[1]: v for k, v in tune_lr.best_params_.items() if 'model__' in k})
print("RF:", {k.split('__')[1]: v for k, v in tune_rf.best_params_.items() if 'model__' in k})
print("GB:", {kimg.split('__')[1]: v for k, v in tune_gb.best_params_.items() if 'model__' in k})

# Define Stacking Classifier using the TUNED *PIPELINES*
estimators = [
    ('lr', best_lr_pipe), 
    ('rf', best_rf_pipe), 
    ('gb', best_gb_pipe)
]

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1,
    passthrough=True # Passes raw predictions to final_estimator
)

# Fit Stacker (on RAW data)
print("\nTraining Stacking Ensemble...")
stack_model.fit(X_train_raw, y_train)

# Final Evaluation Loop
models = {
    "Tuned LR": best_lr_pipe,
    "Tuned RF": best_rf_pipe,
    "Tuned GB": best_gb_pipe,
    "Stacking": stack_model
}

final_results = {}
for name, model in models.items():
    # Evaluate on RAW test data
    preds = model.predict(X_test_raw)
    probs = model.predict_proba(X_test_raw)[:, 1]
    
    # Use labels=[0, 1] for a robust confusion matrix
    cm = confusion_matrix(y_test, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    final_results[name] = {
        "Accuracy": accuracy_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs),
        "Sensitivity (Recall)": recall_score(y_test, preds),
        "Specificity": tn / (tn + fp + 1e-9) # Add epsilon for safety
    }

results_df = pd.DataFrame(final_results).T
print("\n=== FINAL PERFORMANCE (Leakage-Free) ===")
print(results_df.round(3))

# Visualization
results_df[['ROC-AUC', 'F1-Score']].plot(kind='barh', figsize=(8, 5))
plt.title("Tuned Models Performance")
plt.xlabel("Score")
plt.axvline(x=0.5, color='k', linestyle='--', alpha=0.3)
plt.show()